# 线性回归

**教学说明（本科生）：**

本节介绍线性回归的基本原理和实现方法，这是机器学习中最基础的回归算法。

**线性回归基础：**
- **模型形式**：y = w*x + b（单变量）或 y = w1*x1 + w2*x2 + ... + b（多变量）
- **目标**：找到最佳的权重w和偏置b，使得预测值与真实值的误差最小
- **损失函数**：均方误差（MSE）= Σ(y_true - y_pred)^2 / n

**学习内容：**
1. 线性回归的数学原理
2. 梯度下降法求解最优参数
3. 多项式回归处理非线性关系
4. 正则化防止过拟合

**前置知识：**
- 微积分：梯度、偏导数
- 线性代数：向量、矩阵运算
- 概率统计：均方误差的意义

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from IPython.display import display, clear_output
import time

## 构建原始数据

**教学说明（本科生）：**

本节生成用于训练和演示的合成数据集。

**数据生成原理：**
- 使用多项式函数生成真实关系：y = 0.6*x^3 - 1.5*x^2 + 2.0*x + 5.0
- 添加高斯噪声模拟真实数据的随机性
- 生成300个样本点

**为什么使用合成数据？**
1. 可控：可以精确控制数据的生成过程
2. 可视化：便于理解数据的生成机制
3. 验证：可以验证算法是否能恢复真实的函数形式

**学习重点：**
理解数据生成过程对模型训练的影响，体会噪声在机器学习中的作用。

> 💡 **教学提示**：使用合成数据的好处是——我们知道"真实答案"（红色曲线），可以直观地比较模型学到的曲线与真实曲线的差距。

In [ ]:
np.random.seed(42)

n_samples = 300
X = np.linspace(-4, 4, n_samples).reshape(-1, 1)

# True nonlinear function
y_true = 0.6 * X[:, 0]**3 - 1.5 * X[:, 0]**2 + 2.0 * X[:, 0] + 5.0

# Add noise
noise = np.random.normal(0, 4, size=n_samples)
y = y_true + noise

df = pd.DataFrame({
    "x": X[:, 0],
    "y_true": y_true,
    "y_observed": y
})

display(df.head())
print("Dataset shape:", df.shape)

## 原始数据可视化

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.7, label="Observed data")
plt.plot(X, y_true, linewidth=2, label="True function")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic dataset")
plt.legend()
plt.show()

## 划分数据集

**教学说明（本科生）：**

本节将数据集划分为训练集和测试集。

**划分比例：**
- 训练集：70%
- 测试集：30%

**为什么需要划分数据集？**
- **训练集**：用于训练模型，调整参数
- **测试集**：用于评估模型的泛化能力

**学习重点：**
理解模型评估的正确方法，避免在训练数据上评估导致的过拟合误判。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 基础线性回归

**教学说明（本科生）：**

本节使用sklearn的LinearRegression进行基础线性回归。

**线性回归模型：**
- 假设数据满足线性关系：y = w*x + b
- 使用最小二乘法求解最优参数

**评估指标：**
- **MSE**（均方误差）：预测值与真实值差异的平方平均
- **R^2**（决定系数）：模型解释方差的比例，1表示完美拟合，0表示等同于预测均值

**学习重点：**
理解线性回归的基本应用，掌握模型训练和评估的基本流程。

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 评估指标速查

| 指标 | 公式 | 范围 | 含义 |
|------|------|------|------|
| MSE | Σ(y_true-y_pred)²/n | [0, ∞) | 越小越好，受异常值影响大 |
| R² | 1 - MSE/方差 | (-∞, 1] | 越接近1越好，表示模型解释的方差比例 |

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_train_pred = lin_reg.predict(X_train)
y_test_pred = lin_reg.predict(X_test)

print("Coefficient:", lin_reg.coef_[0])
print("Intercept:", lin_reg.intercept_)
print("Train MSE:", mean_squared_error(y_train, y_train_pred))
print("Test MSE:", mean_squared_error(y_test, y_test_pred))
print("Train R^2:", r2_score(y_train, y_train_pred))
print("Test R^2:", r2_score(y_test, y_test_pred))

## 结果可视化

In [ ]:
x_line = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
y_line = lin_reg.predict(x_line)

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, alpha=0.6, label="Train set")
plt.scatter(X_test, y_test, alpha=0.6, label="Test set")
plt.plot(x_line, y_line, linewidth=2, label="Linear regression fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear regression result")
plt.legend()
plt.show()

## 手动定义MSE损失

**教学说明（本科生）：**

本节手动实现均方误差（MSE）损失函数，加深对损失函数的理解。

**MSE公式：**
```
MSE = Σ(y_true - y_pred)^2 / n
```

**梯度计算：**
```
∂MSE/∂w = -2/n * Σ(x * (y_true - y_pred))
∂MSE/∂b = -2/n * Σ(y_true - y_pred)
```

**学习重点：**
理解损失函数的数学形式及其梯度，为实现梯度下降法打下基础。

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

In [ ]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

print("Manual Train MSE:", mse_loss(y_train, y_train_pred))
print("Manual Test MSE:", mse_loss(y_test, y_test_pred))

## 梯度下降

**教学说明（本科生）：**

本节手动实现梯度下降算法训练线性回归模型。

**梯度下降原理：**
```
w = w - learning_rate * ∂loss/∂w
b = b - learning_rate * ∂loss/∂b
```

**关键概念：**
- **学习率**（learning_rate）：控制每次更新的步长
  - 太大：可能震荡甚至发散
  - 太小：收敛速度慢
- **迭代次数**（epochs）：训练的轮数
- **损失下降**：观察损失函数是否持续下降

**可视化学习：**
- 左图：展示拟合直线的动态变化过程
- 右图：展示损失函数的下降曲线

**学习重点：**
理解梯度下降的核心思想——沿着梯度的反方向更新参数以最小化损失。

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, clear_output

In [ ]:
# Use the full dataset for manual gradient descent
x_gd = X[:, 0]
y_gd = y

# Initialize parameters
w = 0.0
b = 0.0

# Hyperparameters
learning_rate = 0.003
epochs = 300
n = len(x_gd)

loss_history = []
w_history = []
b_history = []

# Turn on interactive plotting
plt.ion()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for epoch in range(epochs):
    # Prediction
    y_pred = w * x_gd + b
    
    # Loss
    loss = np.mean((y_gd - y_pred) ** 2)
    loss_history.append(loss)
    w_history.append(w)
    b_history.append(b)
    
    # Gradients
    dw = (-2 / n) * np.sum(x_gd * (y_gd - y_pred))
    db = (-2 / n) * np.sum(y_gd - y_pred)
    
    # Update parameters
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    # Print progress every 10 epochs
    if epoch % 10 == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        print(f"Epoch {epoch:03d}/{epochs}")
        print(f"w = {w:.4f}, b = {b:.4f}")
        print(f"dw = {dw:.4f}, db = {db:.4f}")
        print(f"Loss = {loss:.4f}")
        time.sleep(0.05)
        
        # Update left plot: fitting line
        axes[0].cla()
        axes[0].scatter(x_gd, y_gd, alpha=0.5, label="Data")
        y_fit = w * x_gd + b
        sort_idx = np.argsort(x_gd)
        axes[0].plot(x_gd[sort_idx], y_fit[sort_idx], linewidth=2, label="Current fit")
        axes[0].set_xlabel("x")
        axes[0].set_ylabel("y")
        axes[0].set_title("Gradient descent fitting process")
        axes[0].legend()
        
        # Update right plot: loss curve
        axes[1].cla()
        axes[1].plot(loss_history, linewidth=2)
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("MSE Loss")
        axes[1].set_title("Loss during training")
        
        plt.tight_layout()
        plt.show()
        plt.pause(0.08)

plt.ioff()
print("Training finished.")
print(f"Final w = {w:.4f}, Final b = {b:.4f}")
print(f"Final loss = {loss_history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Final prediction
y_final = w * x_gd + b

# Sort for a clean line plot
sort_idx = np.argsort(x_gd)

plt.figure(figsize=(8, 5))
plt.scatter(x_gd, y_gd, alpha=0.5, label="Data")
plt.plot(x_gd[sort_idx], y_final[sort_idx], linewidth=2, label="Final fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Final gradient descent fit")
plt.legend()
plt.show()

In [ ]:
# Choose several epochs to visualize
selected_epochs = [0, 9, 29, 59, 99, 149, 199, 299]

plt.figure(figsize=(12, 8))

for i, ep in enumerate(selected_epochs, 1):
    plt.subplot(2, 4, i)

    w_ep = w_history[ep]
    b_ep = b_history[ep]
    y_ep = w_ep * x_gd + b_ep

    sort_idx = np.argsort(x_gd)

    plt.scatter(x_gd, y_gd, alpha=0.4, s=15)
    plt.plot(x_gd[sort_idx], y_ep[sort_idx], linewidth=2)

    plt.title(f"Epoch {ep+1}")
    plt.xlabel("x")
    plt.ylabel("y")

plt.tight_layout()
plt.show()

## 最终梯度下降拟合结果

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

In [ ]:
y_gd_fit = w * X[:, 0] + b

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6, label="Observed data")
sort_idx = np.argsort(X[:, 0])
plt.plot(X[:, 0][sort_idx], y_gd_fit[sort_idx], linewidth=2, label="Gradient descent fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Final manual linear regression fit")
plt.legend()
plt.show()

## 多项式回归辅助函数

In [ ]:
def fit_polynomial_regression(X_train, X_test, y_train, y_test, degree):
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    return model, train_mse, test_mse, train_r2, test_r2

## 比较多项式度数

**教学说明（本科生）：**

本节比较不同多项式度数对模型性能的影响。

**多项式回归：**
通过多项式特征扩展，线性模型可以拟合非线性关系：
```
y = w1*x + w2*x^2 + ... + wd*x^d + b
```

**多项式度数的影响：**
- **度数太小**（如d=1）：模型表达能力不足，欠拟合
- **度数适中**（如d=3-5）：模型能捕捉数据的非线性特征
- **度数太大**（如d=12）：模型过于复杂，过拟合

**学习重点：**
理解模型复杂度与拟合能力的关系，为学习过拟合和正则化打下基础。

### 多项式复杂度 vs 拟合效果

| 度数 | 模型复杂度 | 训练误差 | 测试误差 | 现象 |
|------|-----------|---------|---------|------|
| d=1 | 低 | 高 | 高 | 欠拟合 — 太简单了！ |
| d=3 | 中 | 中 | 中 | 合适 — 刚好捕捉模式 |
| d=12 | 高 | 极低 | 高 | 过拟合 — 记住噪音了！ |

In [ ]:
degrees = [1, 2, 3, 5, 8, 12]

results = []

for d in degrees:
    model, train_mse, test_mse, train_r2, test_r2 = fit_polynomial_regression(
        X_train, X_test, y_train, y_test, degree=d
    )
    results.append([d, train_mse, test_mse, train_r2, test_r2])

results_df = pd.DataFrame(
    results,
    columns=["degree", "train_mse", "test_mse", "train_r2", "test_r2"]
)

display(results_df)

## 绘制多项式拟合曲线

In [ ]:
x_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)

plt.figure(figsize=(12, 8))

for i, d in enumerate([1, 3, 12], 1):
    model, _, _, _, _ = fit_polynomial_regression(
        X_train, X_test, y_train, y_test, degree=d
    )
    y_plot = model.predict(x_plot)

    plt.subplot(2, 2, i)
    plt.scatter(X_train, y_train, alpha=0.5, label="Train set")
    plt.scatter(X_test, y_test, alpha=0.5, label="Test set")
    plt.plot(x_plot, y_plot, linewidth=2, label=f"Degree {d}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Polynomial regression (degree={d})")
    plt.legend()

plt.tight_layout()
plt.show()

## 欠拟合和过拟合曲线

**教学说明（本科生）：**

本节可视化训练误差和测试误差随模型复杂度的变化。

**欠拟合（Underfitting）：**
- **表现**：训练误差和测试误差都很大
- **原因**：模型复杂度不足，无法捕捉数据的内在规律
- **解决**：增加模型复杂度

**过拟合（Overfitting）：**
- **表现**：训练误差很小，测试误差很大
- **原因**：模型过于复杂，过度拟合训练数据的噪声
- **解决**：正则化、增加数据、简化模型

**偏差-方差权衡：**
- **偏差**：模型预测期望与真实值的差异（欠拟合）
- **方差**：模型预测的波动程度（过拟合）

**学习重点：**
理解偏差-方差权衡是机器学习的核心概念，指导模型选择和调参。

### 偏差-方差权衡

| 问题 | 偏差 | 方差 | 表现 |
|------|------|------|------|
| 欠拟合 | 高 🔴 | 低 🟢 | 训练和测试误差都高 |
| 过拟合 | 低 🟢 | 高 🔴 | 训练误差极低，测试误差高 |
| 理想 | 低 🟢 | 低 🟢 | 训练和测试误差都低 |

> 💡 **教学提示**：可以用"考试"来类比——欠拟合就像根本没复习（什么都不懂），过拟合就像背下了所有习题但没理解原理（遇到新题就不会了）。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["degree"], results_df["train_mse"], marker="o", label="Train MSE")
plt.plot(results_df["degree"], results_df["test_mse"], marker="s", label="Test MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("MSE")
plt.title("Model complexity: underfitting vs overfitting")
plt.legend()
plt.show()

## 无正则化的高阶多项式回归

**教学说明（本科生）：**

本节展示高阶多项式回归在没有正则化的情况下的表现，这是典型的过拟合现象。

**过拟合表现：**
- 模型在训练集上拟合得很好（低训练误差）
- 但在测试集上表现很差（高测试误差）
- 模型学习了训练数据的噪声，而非真实规律

**学习重点：**
理解正则化的必要性，为学习岭回归和LASSO打下基础。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
degree = 12

poly_lr = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    LinearRegression()
)

poly_lr.fit(X_train, y_train)
y_train_pred_lr = poly_lr.predict(X_train)
y_test_pred_lr = poly_lr.predict(X_test)

print("High-degree polynomial regression")
print("Train MSE:", mean_squared_error(y_train, y_train_pred_lr))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_lr))
print("Train R^2:", r2_score(y_train, y_train_pred_lr))
print("Test R^2:", r2_score(y_test, y_test_pred_lr))

## 岭回归

**教学说明（本科生）：**

本节介绍岭回归（Ridge Regression），一种常用的正则化方法。

**岭回归：**
- **损失函数**：MSE + α * Σw_i^2
- **正则化项**：L2范数（权重的平方和）
- **作用**：惩罚过大的权重，防止模型过拟合

**正则化参数α：**
- α = 0：等同于普通线性回归
- α 较大：权重被强烈惩罚，模型更简单
- α 较小：接近普通线性回归

**岭回归的特点：**
- 所有特征的权重都不会 exactly 为0（特征不会被剔除）
- 适合特征之间存在多重共线性的情况

**学习重点：**
理解正则化如何通过修改损失函数来控制模型复杂度。

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
ridge_model = make_pipeline(
    PolynomialFeatures(degree=12, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_model.fit(X_train, y_train)
y_train_pred_ridge = ridge_model.predict(X_train)
y_test_pred_ridge = ridge_model.predict(X_test)

print("Ridge regression")
print("Train MSE:", mean_squared_error(y_train, y_train_pred_ridge))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_ridge))
print("Train R^2:", r2_score(y_train, y_train_pred_ridge))
print("Test R^2:", r2_score(y_test, y_test_pred_ridge))

## LASSO回归

**教学说明（本科生）：**

本节介绍LASSO回归（Least Absolute Shrinkage and Selection Operator），另一种正则化方法。

**LASSO回归：**
- **损失函数**：MSE + α * Σ|w_i|
- **正则化项**：L1范数（权重的绝对值之和）
- **作用**：惩罚过大的权重，防止模型过拟合

**LASSO的特点：**
- 能够将某些特征的权重压缩为 exactly 0
- 具有**特征选择**功能
- 适合高维稀疏模型

**岭回归 vs LASSO：**
| 特征 | 岭回归（Ridge） | LASSO |
|--|----------------|--|
| 正则化项 | L2范数（平方和） | L1范数（绝对值和） |
| 特征选择 | 不能 | 能 |
| 解的性质 | 连续、平滑 | 可能不连续 |

**学习重点：**
理解L1和L2正则化的差异，掌握特征选择的思路。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
lasso_model = make_pipeline(
    PolynomialFeatures(degree=12, include_bias=False),
    StandardScaler(),
    Lasso(alpha=0.03, max_iter=20000)
)

lasso_model.fit(X_train, y_train)
y_train_pred_lasso = lasso_model.predict(X_train)
y_test_pred_lasso = lasso_model.predict(X_test)

print("LASSO regression")
print("Train MSE:", mean_squared_error(y_train, y_train_pred_lasso))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_lasso))
print("Train R^2:", r2_score(y_train, y_train_pred_lasso))
print("Test R^2:", r2_score(y_test, y_test_pred_lasso))

## 比较回归结果

**教学说明（本科生）：**

本节可视化不同回归模型的拟合效果。

**对比模型：**
1. **线性回归**：基础线性模型
2. **多项式回归**：高阶多项式（无正则化，可能过拟合）
3. **岭回归**：L2正则化
4. **LASSO**：L1正则化

**可视化要点：**
- 观察不同模型的拟合曲线
- 理解正则化如何平滑拟合曲线
- 对比测试集上的预测效果

**学习重点：**
理解不同模型的拟合特性，掌握过拟合和正则化的直观表现。

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

In [ ]:
x_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)

y_plot_lr = poly_lr.predict(x_plot)
y_plot_ridge = ridge_model.predict(x_plot)
y_plot_lasso = lasso_model.predict(x_plot)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.5, label="Train set")
plt.scatter(X_test, y_test, alpha=0.5, label="Test set")
plt.plot(x_plot, y_plot_lr, linewidth=2, label="Polynomial (degree=12)")
plt.plot(x_plot, y_plot_ridge, linewidth=2, label="Ridge")
plt.plot(x_plot, y_plot_lasso, linewidth=2, label="LASSO")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regularization comparison")
plt.legend()
plt.show()

In [ ]:
# Predictions
y_train_pred_lr = poly_lr.predict(X_train)
y_test_pred_lr = poly_lr.predict(X_test)

y_train_pred_ridge = ridge_model.predict(X_train)
y_test_pred_ridge = ridge_model.predict(X_test)

y_train_pred_lasso = lasso_model.predict(X_train)
y_test_pred_lasso = lasso_model.predict(X_test)

# Common axis range
y_all = np.concatenate([
    y_train, y_test,
    y_train_pred_lr, y_test_pred_lr,
    y_train_pred_ridge, y_test_pred_ridge,
    y_train_pred_lasso, y_test_pred_lasso
])
y_min, y_max = y_all.min(), y_all.max()

models_info = [
    ("Polynomial", y_train_pred_lr, y_test_pred_lr),
    ("Ridge", y_train_pred_ridge, y_test_pred_ridge),
    ("LASSO", y_train_pred_lasso, y_test_pred_lasso),
]

plt.figure(figsize=(15, 4))

for i, (title, y_train_pred_model, y_test_pred_model) in enumerate(models_info, 1):
    plt.subplot(1, 3, i)
    
    plt.scatter(y_train, y_train_pred_model, alpha=0.6, label="Train")
    plt.scatter(y_test, y_test_pred_model, alpha=0.8, label="Test")
    plt.plot([y_min, y_max], [y_min, y_max], linestyle="--", linewidth=2, label="Ideal line")
    
    plt.xlabel("True values")
    plt.ylabel("Predicted values")
    plt.title(title)
    plt.legend()

plt.tight_layout()
plt.show()

## 比较系数

**教学说明（本科生）：**

本节对比不同模型的回归系数。

**观察要点：**
1. **线性回归**：系数大小反映特征重要性
2. **岭回归**：系数整体较小，较为平滑
3. **LASSO**：部分系数为0，实现特征选择

**特征重要性：**
- 系数越大，特征对预测结果的影响越大
- 正系数：正相关
- 负系数：负相关

**学习重点：**
理解系数的物理意义，掌握如何从模型中提取特征重要性信息。

---
### 学习重点

- 理解线性回归的数学原理：y = wx + b
- 掌握梯度下降法的核心思想
- 理解过拟合、欠拟合与正则化的关系
- 区分岭回归(L2)和LASSO(L1)的不同特性

### 梯度下降的一步计算示例

假设当前 w=0, b=0，学习率 lr=0.003，有3个样本：

| 样本 | x | y_true | y_pred=wx+b | 误差 |
|------|---|--------|-------------|------|
| 1 | -4.0 | -18.4 | 0 | -18.4 |
| 2 | 0 | 5.0 | 0 | 5.0 |
| 3 | 4.0 | 48.4 | 0 | 48.4 |

dw = (-2/n) * Σ(x·(y_true-y_pred))
db = (-2/n) * Σ(y_true-y_pred)

> 💡 **教学提示**：梯度下降就像在山上找最低点——通过感知当前坡度（梯度），向最陡的下坡方向迈一步（学习率×梯度）。每次迈步后重新感知坡度，再次迈步，直到到达山谷（最小值）。

### 岭回归 vs LASSO

| 特性 | 岭回归 (Ridge) | LASSO |
|------|---------------|-------|
| 正则化项 | α·Σw² (L2) | α·Σ|w| (L1) |
| 效果 | 系数缩小但不为零 | 系数可被压缩为0 |
| 特征选择 | ❌ 不能 | ✅ 可以 |
| 适用场景 | 特征间有相关性 | 高维稀疏特征 |

> 💡 **教学提示**：想象你在约束一个顽皮的孩子（模型参数）：
- 岭回归：让他呆在圆形围栏里（L2范数），不能跑太远
- LASSO：让他呆在菱形围栏里（L1范数），更容易把他逼到角落（参数=0）

## 模型性能比较

**教学说明（本科生）：**

本节综合对比不同模型的性能。

**评估指标总结：**
- **训练MSE / 测试MSE**：误差越小越好
- **训练R^2 / 测试R^2**：越接近1越好

**模型选择原则：**
1. 优先关注测试性能
2. 训练性能好但测试性能差 → 过拟合
3. 训练和测试性能都差 → 欠拟合
4. 训练和测试性能都好 → 理想情况

**学习重点：**
掌握模型评估和选择的基本方法，理解性能指标的含义。

In [ ]:
summary = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression", "Ridge", "LASSO"],
    "Train MSE": [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_train, y_train_pred_lr),
        mean_squared_error(y_train, y_train_pred_ridge),
        mean_squared_error(y_train, y_train_pred_lasso)
    ],
    "Test MSE": [
        mean_squared_error(y_test, y_test_pred),
        mean_squared_error(y_test, y_test_pred_lr),
        mean_squared_error(y_test, y_test_pred_ridge),
        mean_squared_error(y_test, y_test_pred_lasso)
    ],
    "Train R^2": [
        r2_score(y_train, y_train_pred),
        r2_score(y_train, y_train_pred_lr),
        r2_score(y_train, y_train_pred_ridge),
        r2_score(y_train, y_train_pred_lasso)
    ],
    "Test R^2": [
        r2_score(y_test, y_test_pred),
        r2_score(y_test, y_test_pred_lr),
        r2_score(y_test, y_test_pred_ridge),
        r2_score(y_test, y_test_pred_lasso)
    ]
})

display(summary)

# KNN算法

**教学说明（本科生）：**

本节介绍K近邻（K-Nearest Neighbors, KNN）分类算法。

**KNN算法原理：**
1. 给定一个待分类样本
2. 找出训练集中距离最近的K个样本
3. 根据这K个邻居的类别进行投票，决定待分类样本的类别

**核心要素：**
- **K值**：邻居数量，是重要的超参数
- **距离度量**：常用欧氏距离
- **投票规则**：多数表决

**算法特点：**
- **惰性学习**（Lazy Learning）：没有显式的训练过程
- **非参数方法**：不对数据分布做假设
- **适合多分类**：自然支持多类别

**学习重点：**
理解KNN的基本思想，掌握超参数对模型性能的影响。

---
### 学习重点

- 理解KNN的"近朱者赤"核心思想
- 掌握k值对模型复杂度的影响
- 理解标准化对基于距离的算法的必要性
- 理解决策边界的含义

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 创建数据集

**教学说明（本科生）：**

本节生成用于KNN分类的合成数据集。

**make_moons数据集：**
- 生成两个交错的月牙形类别
- 数据非线性可分
- 适合测试分类算法的性能

**学习重点：**
理解非线性可分数据的特点，体会KNN在这种数据上的表现。

In [ ]:
np.random.seed(42)

X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

df = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "label": y
})

display(df.head())
print("Dataset shape:", df.shape)

## 可视化原始数据

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.8)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Synthetic dataset for KNN classification")
plt.show()

## 分割数据集

**教学说明（本科生）：**

本节将数据集划分为训练集和测试集。

**stratify参数：**
- 保持训练集和测试集中各类别的比例与原始数据一致
- 避免划分后类别分布不均衡

**学习重点：**
理解分层抽样在分类任务中的重要性。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 数据标准化

**教学说明（本科生）：**

本节对数据进行标准化处理。

**标准化公式：**
```
x_scaled = (x - mean) / std
```

**为什么需要标准化？**
- KNN使用距离度量，特征量纲不同会影响距离计算
- 标准化后，所有特征对距离的贡献更加均衡

**学习重点：**
理解特征缩放在基于距离的算法中的重要性。

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("First 5 rows before scaling:")
print(X_train[:5])

print("\nFirst 5 rows after scaling:")
print(X_train_scaled[:5])

## 训练KNN

**教学说明（本科生）：**

本节训练KNN分类器。

**KNN参数：**
- **n_neighbors**（k值）：邻居数量，是关键超参数
  - k太小：对噪声敏感，容易过拟合
  - k太大：决策边界过于平滑，可能欠拟合
- **权重**：uniform（等权）或 distance（距离加权）
- **距离度量**：默认欧氏距离

**学习重点：**
掌握KNN模型的训练方法，理解k值对模型性能的影响。

In [ ]:
k = 5
knn = KNeighborsClassifier(n_neighbors=k)

knn.fit(X_train_scaled, y_train)

y_train_pred = knn.predict(X_train_scaled)
y_test_pred = knn.predict(X_test_scaled)

print(f"k = {k}")
print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy:", accuracy_score(y_test, y_test_pred))

## 分类性能

In [ ]:
results_df = pd.DataFrame({
    "True label": y_test,
    "Predicted label": y_test_pred
})

display(results_df.head(15))

## 结果可视化

In [ ]:
correct = y_test == y_test_pred
incorrect = y_test != y_test_pred

plt.figure(figsize=(8, 6))
plt.scatter(X_test[correct, 0], X_test[correct, 1], c=y_test[correct], alpha=0.7, label="Correct")
plt.scatter(X_test[incorrect, 0], X_test[incorrect, 1], c=y_test[incorrect], marker="x", s=100, label="Incorrect")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Test set predictions")
plt.legend()
plt.show()

## 混淆矩阵

**教学说明（本科生）：**

本节展示分类结果的混淆矩阵和分类报告。

**混淆矩阵：**
| | 预测0 | 预测1 |
|--|-------|-------|
| **真实0** | TN（真负） | FP（假正） |
| **真实1** | FN（假负） | TP（真正） |

**评估指标：**
- **准确率**（Accuracy）：(TP + TN) / 总数
- **精确率**（Precision）：TP / (TP + FP)
- **召回率**（Recall）：TP / (TP + FN)
- **F1分数**：2 * Precision * Recall / (Precision + Recall)

**学习重点：**
理解分类评估指标，掌握混淆矩阵的解读方法。

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=["True 0", "True 1"], columns=["Pred 0", "Pred 1"])
display(cm_df)

print(classification_report(y_test, y_test_pred))

## 尝试不同的k值

**教学说明（本科生）：**

本节系统地尝试不同的k值，找到最优的邻居数量。

**超参数搜索：**
- 遍历不同的k值（1-30）
- 记录每个k值下的训练和测试准确率
- 选择测试准确率最高的k值

**学习重点：**
掌握超参数调优的基本方法，理解k值对模型性能的影响。

### k值选择指南

| k值 | 决策边界 | 模型复杂度 | 风险 |
|-----|---------|-----------|------|
| k=1 | 非常曲折 | 极高 | 过拟合 — 对噪声敏感 |
| k=5~10 | 较平滑 | 适中 | 泛化能力好 ✅ |
| k=25+ | 过于平滑 | 低 | 欠拟合 — 丢失细节 |

经验法则：k通常取奇数（避免平局），且不超过√n（n为样本数）。

In [ ]:
k_values = range(1, 31)
train_acc_list = []
test_acc_list = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    
    y_train_pred_k = model.predict(X_train_scaled)
    y_test_pred_k = model.predict(X_test_scaled)
    
    train_acc_list.append(accuracy_score(y_train, y_train_pred_k))
    test_acc_list.append(accuracy_score(y_test, y_test_pred_k))

results_k = pd.DataFrame({
    "k": list(k_values),
    "train_accuracy": train_acc_list,
    "test_accuracy": test_acc_list
})

display(results_k.head(10))

## 过拟合和欠拟合

**教学说明（本科生）：**

本节可视化不同k值下的模型性能变化。

**KNN中的过拟合和欠拟合：**
- **k=1**：决策边界非常曲折，过拟合训练数据
- **k适中**：决策边界平滑，泛化能力好
- **k太大**：决策边界过于平滑，欠拟合

**学习重点：**
理解k值如何影响模型复杂度，掌握过拟合和欠拟合在KNN中的表现。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, train_acc_list, marker="o", label="Train accuracy")
plt.plot(k_values, test_acc_list, marker="s", label="Test accuracy")
plt.xlabel("Number of neighbors (k)")
plt.ylabel("Accuracy")
plt.title("KNN model complexity: underfitting vs overfitting")
plt.legend()
plt.show()

## 最佳k值

**教学说明（本科生）：**

本节找到最优的k值并进行解释。

**选择最佳k值的考虑：**
1. 测试准确率最高
2. 模型复杂度适中
3. 交叉验证稳定性

**经验法则：**
- k通常取奇数（避免投票平局）
- k不超过sqrt(n)（样本数量的平方根）

**学习重点：**
理解模型选择的原则，掌握如何在性能和复杂度之间权衡。

In [ ]:
best_k = results_k.loc[results_k["test_accuracy"].idxmax(), "k"]
best_test_acc = results_k["test_accuracy"].max()

print(f"Best k based on test accuracy: {best_k}")
print(f"Best test accuracy: {best_test_acc:.4f}")

print("\nInterpretation:")
print("- Very small k may lead to overfitting.")
print("- Very large k may lead to underfitting.")
print("- A moderate k often gives the best generalization.")

## 决策边界可视化

**教学说明（本科生）：**

本节定义决策边界可视化函数。

**决策边界：**
- 分类算法在特征空间中划分不同类别区域的边界
- 决策边界越复杂，模型越容易过拟合
- 决策边界越平滑，模型越容易欠拟合

**学习重点：**
理解决策边界的含义，掌握如何通过可视化观察模型的决策能力。

### 如何解读决策边界图

- **不同颜色区域**：模型认为属于不同类别的区域
- **边界线**：模型犹豫不决的地方（两类概率相等）
- **边界曲直**：k=1时边界非常曲折（过拟合），k=25时边界过于平滑（欠拟合）
- **正确/错误标记**：圆圈为正确，叉号为错误

> 💡 **教学提示**：决策边界是模型的"世界观"——它画出了模型认为的类别分界线。边界越复杂，模型越"多疑"（过拟合）；边界越平滑，模型越"粗线条"（欠拟合）。

In [ ]:
for k in [1, 5, 10, 25]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    plot_decision_boundary(
        model,
        X_train_scaled,
        y_train,
        scaler,
        title=f"KNN decision boundary (k = {k})"
    )

In [ ]:
best_model = KNeighborsClassifier(n_neighbors=int(best_k))
best_model.fit(X_train_scaled, y_train)

y_train_best = best_model.predict(X_train_scaled)
y_test_best = best_model.predict(X_test_scaled)

print("Best model performance")
print("Train accuracy:", accuracy_score(y_train, y_train_best))
print("Test accuracy:", accuracy_score(y_test, y_test_best))

plot_decision_boundary(
    best_model,
    X_train_scaled,
    y_train,
    scaler,
    title=f"Best KNN decision boundary (k = {best_k})"
)

# 多项式回归的超参数优化（K-fold + 粒子群算法）

**教学说明（本科生）：**

本节介绍如何结合K折交叉验证和粒子群算法优化多项式回归的超参数。

**超参数优化：**
多项式回归有两个重要超参数：
1. **degree**（多项式度数）：控制模型复杂度
2. **alpha**（正则化参数）：控制正则化强度

**K折交叉验证：**
1. 将训练数据分成K份
2. 每次用K-1份训练，1份验证
3. 计算K次验证的平均误差

**PSO优化：**
- 用粒子群算法搜索最优的(degree, alpha)组合
- 适应度函数：K折交叉验证的平均误差

**学习重点：**
理解超参数优化的完整流程，掌握组合优化方法的应用。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

## 定义 K-fold + 多项式 Ridge 的目标函数

**教学说明（本科生）：**

本节定义超参数优化的目标函数（评估函数）。

**目标函数：**
- 输入：(degree, alpha)
- 输出：K折交叉验证的平均MSE

**K折交叉验证的作用：**
- 更稳健的性能评估
- 减少对单一训练/测试划分的依赖

**学习重点：**
理解如何将交叉验证与优化算法结合，实现稳健的超参数搜索。

In [ ]:
def evaluate_polynomial_ridge_cv(X_train, y_train, degree, alpha, n_splits=5, random_state=42):
    """
    Evaluate polynomial ridge regression using K-fold cross-validation.
    Return mean CV MSE.
    """
    degree = int(round(degree))
    degree = max(1, degree)
    alpha = max(1e-6, alpha)

    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        Ridge(alpha=alpha)
    )

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # cross_val_score with negative MSE
    neg_mse_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring="neg_mean_squared_error"
    )

    mean_mse = -np.mean(neg_mse_scores)
    return mean_mse

## 检查目标函数是否正常

In [ ]:
test_degree = 3
test_alpha = 1.0

test_cv_mse = evaluate_polynomial_ridge_cv(
    X_train, y_train,
    degree=test_degree,
    alpha=test_alpha,
    n_splits=5
)

print(f"Test degree = {test_degree}")
print(f"Test alpha = {test_alpha}")
print(f"5-fold CV MSE = {test_cv_mse:.4f}")

## 定义一个PSO函数

**教学说明（本科生）：**

本节实现粒子群优化算法用于超参数搜索。

**PSO用于超参数优化：**
- 每个粒子表示一个(degree, alpha)组合
- 适应度：K折交叉验证的平均误差
- 目标：找到使误差最小的(degree, alpha)组合

**优化过程：**
1. 初始化粒子位置和速度
2. 迭代更新粒子位置和速度
3. 记录个体最优和全局最优
4. 输出全局最优解

**学习重点：**
理解PSO在连续空间优化中的应用，掌握超参数优化的实现细节。

In [ ]:
def particle_swarm_optimization(
    objective_function,
    bounds,
    n_particles=12,
    n_iterations=20,
    w=0.7,
    c1=1.5,
    c2=1.5,
    random_state=42
):
    """
    Simple PSO for 2D hyperparameter optimization.
    bounds: [(min1, max1), (min2, max2)]
    """
    np.random.seed(random_state)

    dim = len(bounds)

    # Initialize particle positions and velocities
    positions = np.array([
        [np.random.uniform(low, high) for (low, high) in bounds]
        for _ in range(n_particles)
    ])

    velocities = np.random.uniform(-1, 1, size=(n_particles, dim))

    # Personal best
    pbest_positions = positions.copy()
    pbest_scores = np.array([objective_function(pos) for pos in positions])

    # Global best
    gbest_index = np.argmin(pbest_scores)
    gbest_position = pbest_positions[gbest_index].copy()
    gbest_score = pbest_scores[gbest_index]

    history = []

    for iteration in range(n_iterations):
        for i in range(n_particles):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - positions[i])
                + c2 * r2 * (gbest_position - positions[i])
            )

            positions[i] = positions[i] + velocities[i]

            # Apply bounds
            for d in range(dim):
                low, high = bounds[d]
                positions[i, d] = np.clip(positions[i, d], low, high)

            # Evaluate
            score = objective_function(positions[i])

            # Update personal best
            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = positions[i].copy()

            # Update global best
            if score < gbest_score:
                gbest_score = score
                gbest_position = positions[i].copy()

        history.append(gbest_score)

        print(
            f"Iteration {iteration+1:02d}/{n_iterations} | "
            f"Best CV MSE = {gbest_score:.4f} | "
            f"Best degree = {int(round(gbest_position[0]))} | "
            f"Best alpha = {gbest_position[1]:.6f}"
        )

    return gbest_position, gbest_score, history

## 定义 PSO 的优化目标
- degree
- alpha

In [ ]:
def pso_objective(params):
    degree = params[0]
    alpha = params[1]

    cv_mse = evaluate_polynomial_ridge_cv(
        X_train, y_train,
        degree=degree,
        alpha=alpha,
        n_splits=5,
        random_state=42
    )
    return cv_mse

## 运行 PSO 超参数优化

**教学说明（本科生）：**

本节运行PSO算法进行超参数优化。

**优化结果分析：**
1. **最优degree**：多项式回归的最佳复杂度
2. **最优alpha**：正则化的最佳强度
3. **最优CV MSE**：交叉验证的最小误差

**学习重点：**
理解超参数优化的完整流程，掌握如何解读优化结果。

In [ ]:
bounds = [
    (1, 12),        # degree
    (0.0001, 10.0)  # alpha
]

best_position, best_score, pso_history = particle_swarm_optimization(
    objective_function=pso_objective,
    bounds=bounds,
    n_particles=15,
    n_iterations=25,
    w=0.7,
    c1=1.5,
    c2=1.5,
    random_state=42
)

best_degree = int(round(best_position[0]))
best_alpha = best_position[1]

print("\nOptimization finished.")
print(f"Best degree = {best_degree}")
print(f"Best alpha = {best_alpha:.6f}")
print(f"Best 5-fold CV MSE = {best_score:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(pso_history, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Best CV MSE")
plt.title("PSO optimization process")
plt.show()

## 使用最优参数重新训练模型

**教学说明（本科生）：**

本节使用PSO找到的最优参数重新训练模型，并在测试集上评估。

**完整流程：**
1. 用PSO找到最优(degree, alpha)
2. 用最优参数训练最终模型
3. 在测试集上评估泛化性能

**学习重点：**
理解超参数优化与模型训练的完整流程，掌握如何应用优化结果。

In [ ]:
best_poly_ridge_model = make_pipeline(
    PolynomialFeatures(degree=best_degree, include_bias=False),
    StandardScaler(),
    Ridge(alpha=best_alpha)
)

best_poly_ridge_model.fit(X_train, y_train)

y_train_pred_best = best_poly_ridge_model.predict(X_train)
y_test_pred_best = best_poly_ridge_model.predict(X_test)

print("Optimized polynomial ridge regression")
print("Train MSE:", mean_squared_error(y_train, y_train_pred_best))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_best))
print("Train R^2:", r2_score(y_train, y_train_pred_best))
print("Test R^2:", r2_score(y_test, y_test_pred_best))

In [ ]:
x_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
y_plot_best = best_poly_ridge_model.predict(x_plot)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.5, label="Train set")
plt.scatter(X_test, y_test, alpha=0.5, label="Test set")
plt.plot(x_plot, y_plot_best, linewidth=2, label=f"Optimized model (degree={best_degree}, alpha={best_alpha:.3f})")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Optimized polynomial ridge regression")
plt.legend()
plt.show()

In [ ]:
y_all = np.concatenate([y_test, y_test_pred_best])
y_min, y_max = y_all.min(), y_all.max()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_pred_best, alpha=0.8, label="Test samples")
plt.plot([y_min, y_max], [y_min, y_max], linestyle="--", linewidth=2, label="y = x")
plt.xlabel("True values")
plt.ylabel("Predicted values")
plt.title("Optimized model: True vs Predicted")
plt.xlim(y_min, y_max)
plt.ylim(y_min, y_max)
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Polynomial Regression",
        "Ridge",
        "LASSO",
        "Optimized Polynomial Ridge (PSO + KFold)"
    ],
    "Train MSE": [
        mean_squared_error(y_train, y_train_pred_lr),
        mean_squared_error(y_train, y_train_pred_ridge),
        mean_squared_error(y_train, y_train_pred_lasso),
        mean_squared_error(y_train, y_train_pred_best)
    ],
    "Test MSE": [
        mean_squared_error(y_test, y_test_pred_lr),
        mean_squared_error(y_test, y_test_pred_ridge),
        mean_squared_error(y_test, y_test_pred_lasso),
        mean_squared_error(y_test, y_test_pred_best)
    ],
    "Train R^2": [
        r2_score(y_train, y_train_pred_lr),
        r2_score(y_train, y_train_pred_ridge),
        r2_score(y_train, y_train_pred_lasso),
        r2_score(y_train, y_train_pred_best)
    ],
    "Test R^2": [
        r2_score(y_test, y_test_pred_lr),
        r2_score(y_test, y_test_pred_ridge),
        r2_score(y_test, y_test_pred_lasso),
        r2_score(y_test, y_test_pred_best)
    ]
})

display(comparison_df)